# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library. The dataset contains quantitative and metadata tables from human mass spectrometry experiments, made available via the Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL. All references to entities (record sets, fields, columns) use their unique `@id` as defined in the schema.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print metadata overview
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Identifier: {getattr(meta, 'identifier', None)}\nPublished: {getattr(meta, 'datePublished', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant structure. Entities are referenced by their `@id`.

In [ ]:
# List available record set @ids with descriptions

record_sets = list(dataset.record_sets)
print('Available Record Sets (by @id):')
for rec_set in record_sets:
    print(f"- @id: {rec_set['@id']}")
    print(f"  Name: {rec_set.get('name','')} | Description: {rec_set.get('description','')}")

# For one record set, show the fields (column/headings)
if record_sets:
    chosen_record_set_id = record_sets[0]['@id']
    print(f"\nFields in record set '{chosen_record_set_id}':")
    for field in record_sets[0].get('field', []):
        print(f"  - @id: {field['@id']}, name: {field.get('name','')}, description: {field.get('description','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data for all record sets

dataframes = {}
for rec_set in record_sets:
    rec_id = rec_set['@id']
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded record set: {rec_id}, shape: {df.shape}")
    except Exception as e:
        print(f"Could not load record set {rec_id}: {e}")

# Show column names for the first record set
if record_sets:
    print(f"\nColumns in record set '{chosen_record_set_id}':")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping. All columns/fields referenced by their schema `@id`.

In [ ]:
# Example: filter, normalize, and group by a key field in the main protein table

# User: adjust these to match the actual @id and field names as seen above, e.g.:
# For this example, let's use typical fields such as abundance or MW (Molecular Weight).

# Modify these according to the actual @ids printed in the overview above:
record_set_id = chosen_record_set_id  # Use the main record set

# Example field picking logic:
main_df = dataframes[record_set_id]
available_fields = list(main_df.columns)
print(f"Available fields: {available_fields}")

# Try common possibilities
numeric_field_id = None
for candidate in ['cr:abundance', 'cr:MW', 'MW', 'cr:MW (kDa)', 'cr:abundance (norm)']:
    if candidate in available_fields:
        numeric_field_id = candidate
        break
# Fallback to first numeric column
if numeric_field_id is None:
    for c in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[c]):
            numeric_field_id = c
            break

if numeric_field_id is not None:
    print(f"\nChosen numeric field for EDA: {numeric_field_id}")
    # Filter: only retain rows with value > threshold
    threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}\n")

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical field (e.g. 'cr:sample', 'cr:modification', etc)
    group_field_id = None
    for candidate in ['cr:sample', 'cr:modification', 'cr:protein', 'cr:accession']:
        if candidate in available_fields:
            group_field_id = candidate
            break

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
        print(f"\nMean {numeric_field_id} by {group_field_id} (first 5):")
        print(grouped_df.head())
else:
    print("No suitable numeric field for EDA found in this dataset.")

## 5. Visualization
Visualize the distribution or comparison of the normalized numeric field (from above EDA).

In [ ]:
# Simple histogram of values (only if numeric_field_id available)
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If normalization done
    col_norm = f"{numeric_field_id}_normalized"
    if col_norm in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[col_norm].dropna(), kde=True, color='mediumpurple')
        plt.title(f"Normalized Distribution of {numeric_field_id} (filtered)")
        plt.xlabel(f"{numeric_field_id} (normalized)")
        plt.show()

## 6. Conclusion

In this notebook, we explored a FAIRⱼ-compliant Croissant dataset for human protein mass spectrometry. We loaded metadata and record sets using their `@id`, reviewed available fields, loaded the main table, performed basic filtering and normalization, and visualized numeric data distributions. Dataset exploration with `mlcroissant` supports reproducible, structured, and schema-driven scientific data analysis workflows, making large-scale data easier to understand and process.